# Expansion Prediction - Gold Layer

Predicts sales for expansion candidates and applies business constraints.

**Key Logic:**
1. Sanity check against existing store metrics
2. Heuristic sales prediction (placeholder for ML model)
3. Apply business constraints (min sales, population)
4. Exclude H3 cells covered by existing LCE isochrones
5. Rank candidates by predicted sales

**Inputs:**
- `{catalog}.{silver_schema}.expansion_candidates_h3`
- `{catalog}.{silver_schema}.isochrones_lce`
- `{catalog}.{silver_schema}.existing_stores_h3`

**Output:**
- `{catalog}.{gold_schema}.expansion_candidates_h3_enhanced`

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, expr, explode, lit
from pyspark.sql.window import Window

dbutils.widgets.text("catalog", "jdub_demo_aws")
dbutils.widgets.text("silver_schema", "geo_silver")
dbutils.widgets.text("gold_schema", "geo_gold")
dbutils.widgets.text("min_predicted_sales", "250000")
dbutils.widgets.text("min_population", "5000")

catalog = dbutils.widgets.get("catalog")
silver_schema = dbutils.widgets.get("silver_schema")
gold_schema = dbutils.widgets.get("gold_schema")
min_predicted_sales = int(dbutils.widgets.get("min_predicted_sales"))
min_population = int(dbutils.widgets.get("min_population"))

# Table names
candidates_table = f"{catalog}.{silver_schema}.expansion_candidates_h3"
isochrones_table = f"{catalog}.{silver_schema}.isochrones_lce"
existing_stores_table = f"{catalog}.{silver_schema}.existing_stores_h3"
output_table = f"{catalog}.{gold_schema}.expansion_candidates_h3_enhanced"

print(f"Candidates: {candidates_table}")
print(f"Existing stores: {existing_stores_table}")
print(f"Output: {output_table}")
print(f"Min predicted sales: ${min_predicted_sales:,}")
print(f"Min population: {min_population:,}")

In [ ]:
# MAGIC %md
# MAGIC ## Load Data

In [ ]:
# Load expansion candidates
candidates = spark.table(candidates_table)
print(f"Loaded {candidates.count()} expansion candidates")

# Load existing store metrics for baseline comparison
try:
    existing_stores = spark.table(existing_stores_table)
    
    # Calculate total POI count by summing POI category columns
    existing_with_total_poi = existing_stores.withColumn(
        "total_poi_count",
        F.coalesce(col("total_retail_pois"), lit(0)) +
        F.coalesce(col("total_food_drink_pois"), lit(0)) +
        F.coalesce(col("total_leisure_pois"), lit(0)) +
        F.coalesce(col("total_education_pois"), lit(0)) +
        F.coalesce(col("total_healthcare_pois"), lit(0)) +
        F.coalesce(col("total_financial_pois"), lit(0)) +
        F.coalesce(col("total_tourism_pois"), lit(0)) +
        F.coalesce(col("total_transportation_pois"), lit(0))
    )
    
    existing_metrics = existing_with_total_poi.agg(
        F.avg("population").alias("avg_population"),
        F.avg("total_poi_count").alias("avg_poi_count"),
        F.min("population").alias("min_population"),
        F.max("population").alias("max_population")
    ).collect()[0]
    
    print(f"\nExisting LCE Store Baseline:")
    print(f"  Avg population: {existing_metrics['avg_population']:,.0f}")
    print(f"  Avg POI count: {existing_metrics['avg_poi_count']:,.0f}")
    print(f"  Population range: {existing_metrics['min_population']:,.0f} - {existing_metrics['max_population']:,.0f}")
except Exception as e:
    print(f"Could not load existing store metrics: {e}")
    existing_metrics = None

In [ ]:
# MAGIC %md
# MAGIC ## Exclude Existing Store Trade Areas

In [ ]:
# Get H3 cells covered by existing LCE store isochrones
try:
    lce_isochrones = spark.table(isochrones_table)
    
    # Generate H3 cells for each existing store's trade area
    existing_h3_cells = lce_isochrones.select(
        explode(expr("h3_coverash3string(ST_AsBinary(geometry), 8)")).alias("h3_cell_id")
    ).distinct()
    
    existing_cell_count = existing_h3_cells.count()
    print(f"H3 cells covered by existing stores: {existing_cell_count:,}")
    
    # Exclude candidates in existing trade areas
    candidates_filtered = candidates.join(
        existing_h3_cells,
        candidates["h3_cell_id"] == existing_h3_cells["h3_cell_id"],
        "left_anti"
    )
    
    excluded_count = candidates.count() - candidates_filtered.count()
    print(f"Candidates excluded (in existing trade areas): {excluded_count:,}")
    print(f"Candidates remaining: {candidates_filtered.count():,}")
    
except Exception as e:
    print(f"Could not exclude existing trade areas: {e}")
    candidates_filtered = candidates

In [ ]:
# MAGIC %md
# MAGIC ## Predict Sales (Heuristic Model)

In [ ]:
# Heuristic sales prediction formula
# This is a placeholder for a real ML model
candidates_with_sales = candidates_filtered.withColumn(
    "predicted_annual_sales",
    (
        # Base sales
        lit(300000) +
        
        # Target demographic impact (young adults 20-34)
        F.coalesce(col("target_demographic"), lit(0)) * 30 +
        
        # POI impact (commercial vibrancy)
        F.coalesce(col("total_poi"), lit(0)) * 50 +
        
        # Population impact
        F.coalesce(col("population"), lit(0)) * 2
    ).cast("long")
).withColumn(
    "predicted_monthly_sales",
    (col("predicted_annual_sales") / 12).cast("long")
)

print("Sales prediction formula applied")
display(candidates_with_sales.select(
    "h3_cell_id", "urbanicity", "population", "target_demographic", 
    "total_poi", "predicted_annual_sales"
).orderBy(F.desc("predicted_annual_sales")).limit(10))

In [ ]:
# MAGIC %md
# MAGIC ## Apply Business Constraints

In [ ]:
# Apply minimum thresholds
candidates_constrained = candidates_with_sales.filter(
    (col("predicted_annual_sales") >= min_predicted_sales) &
    (col("population") >= min_population)
)

before_count = candidates_with_sales.count()
after_count = candidates_constrained.count()
print(f"Before constraints: {before_count:,}")
print(f"After constraints: {after_count:,}")
print(f"Filtered out: {before_count - after_count:,}")

In [ ]:
# MAGIC %md
# MAGIC ## Rank Candidates

In [ ]:
# Add ranking
window_spec = Window.orderBy(F.desc("predicted_annual_sales"))

candidates_ranked = candidates_constrained.withColumn(
    "rank", F.row_number().over(window_spec)
).withColumn(
    "percentile", F.percent_rank().over(window_spec)
)

print("Top 20 expansion candidates:")
display(candidates_ranked.select(
    "rank", "h3_cell_id", "latitude", "longitude", "urbanicity",
    "population", "target_demographic", "total_poi",
    "predicted_annual_sales", "predicted_monthly_sales"
).limit(20))

In [ ]:
# MAGIC %md
# MAGIC ## Write to Gold

In [ ]:
# Add processing timestamp and metadata
candidates_final = candidates_ranked.withColumn(
    "processing_timestamp", F.current_timestamp()
).withColumn(
    "min_sales_threshold", lit(min_predicted_sales)
).withColumn(
    "min_population_threshold", lit(min_population)
)

# Write to gold
(
    candidates_final
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(output_table)
)

print(f"\nWritten {candidates_final.count():,} ranked expansion candidates to {output_table}")

In [ ]:
# MAGIC %md
# MAGIC ## Summary Statistics

In [ ]:
print("Expansion Candidates Summary:")
display(spark.sql(f"""
  SELECT
    COUNT(*) as total_candidates,
    ROUND(AVG(predicted_annual_sales), 0) as avg_predicted_sales,
    ROUND(MIN(predicted_annual_sales), 0) as min_predicted_sales,
    ROUND(MAX(predicted_annual_sales), 0) as max_predicted_sales,
    ROUND(AVG(population), 0) as avg_population,
    ROUND(AVG(target_demographic), 0) as avg_target_demo,
    ROUND(AVG(total_poi), 0) as avg_poi_count
  FROM {output_table}
"""))

print("\nBy Urbanicity:")
display(spark.sql(f"""
  SELECT
    urbanicity,
    COUNT(*) as count,
    ROUND(AVG(predicted_annual_sales), 0) as avg_sales,
    ROUND(AVG(population), 0) as avg_pop
  FROM {output_table}
  GROUP BY urbanicity
  ORDER BY avg_sales DESC
"""))